# Parte 1 — Treinamento do modelo de comparação (CNN)

Treina uma rede convolucional pré-treinada (torchvision) nas mesmas imagens e divisão usadas no YOLO.
Ao final salva, em `runs_comparacao/`, o melhor peso (`<modelo>_best.pt`) e um arquivo de metadados (`<modelo>_meta.json`) que o notebook de comparação lê automaticamente.

**Próximo passo:** abrir `2_Comparacao_YOLO_vs_CNN.ipynb`.

In [1]:
!pip install -q torch torchvision matplotlib pandas

In [ ]:
import os, time, json, random, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ============ CONFIGURAÇÕES ============
PASTA_DADOS = "Tumor-Cerebral-1"   # mesma pasta do YOLO (com train/valid/test)
MODELO      = "efficientnet_b0"    # "efficientnet_b0" | "resnet50" | "convnext_tiny"
IMG_SIZE    = 512                  # mesmo ALVO do YOLO. Se estiver lento na CPU, use 256 (e avise na comparação)
BATCH       = 32
EPOCAS      = 40
LR          = 3e-4
WEIGHT_DECAY= 0.01
PACIENCIA   = 10                   # early stopping (épocas sem melhora na validação)
EPOCAS_CABECA = 2                  # épocas iniciais treinando só a camada final (backbone congelado)
NUM_WORKERS = 2                    # se der erro no Windows, coloque 0
SEED        = 42
PASTA_SAIDA = "runs"
# =======================================

def selecionar_device():
    if torch.backends.mps.is_available(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")

def fixar_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

fixar_seed(SEED)
device = selecionar_device()
os.makedirs(PASTA_SAIDA, exist_ok=True)
print("Device:", device)

## 1. Dados
Lê as pastas `train`, `valid` e `test`; cada subpasta é uma classe (`notumor`, `pituitary`, ...). As augmentations imitam as do YOLO (flip horizontal, pequena escala/translação, brilho leve e random erasing).

In [ ]:
media, desvio = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

tf_treino = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.7, 1.3)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(media, desvio),
    transforms.RandomErasing(p=0.1),
])
tf_aval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(media, desvio),
])

ds_treino = datasets.ImageFolder(os.path.join(PASTA_DADOS, "train"), tf_treino)
ds_val    = datasets.ImageFolder(os.path.join(PASTA_DADOS, "valid"), tf_aval)
ds_teste  = datasets.ImageFolder(os.path.join(PASTA_DADOS, "test"),  tf_aval)

assert ds_treino.classes == ds_val.classes == ds_teste.classes, "As classes diferem entre train/valid/test!"
CLASSES = ds_treino.classes
print("Classes:", CLASSES)
print(f"train={len(ds_treino)} | valid={len(ds_val)} | test={len(ds_teste)}")

kw = dict(batch_size=BATCH, num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))
dl_treino = DataLoader(ds_treino, shuffle=True,  **kw)
dl_val    = DataLoader(ds_val,    shuffle=False, **kw)
dl_teste  = DataLoader(ds_teste,  shuffle=False, **kw)

## 2. Modelo
Rede pré-treinada no ImageNet, com a última camada trocada para o número de classes do seu problema.

In [ ]:
def criar_modelo(nome, n_classes):
    if nome == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        cabeca = m.classifier
    elif nome == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, n_classes))
        cabeca = m.fc
    elif nome == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n_classes)
        cabeca = m.classifier
    else:
        raise ValueError(nome)
    return m, cabeca

def congelar_backbone(modelo, cabeca, congelar):
    for p in modelo.parameters(): p.requires_grad = not congelar
    for p in cabeca.parameters(): p.requires_grad = True

modelo, cabeca = criar_modelo(MODELO, len(CLASSES))
modelo = modelo.to(device)
n_params = sum(p.numel() for p in modelo.parameters())
print(f"{MODELO}: {n_params/1e6:.2f} M parâmetros")

## 3. Treinamento
AdamW + cosine annealing, early stopping pela acurácia de validação. As primeiras épocas treinam só a camada final (equivalente ao `freeze` do YOLO) e depois toda a rede é liberada.

In [ ]:
def rodar_epoca(modelo, dl, criterio, otimizador=None):
    treinando = otimizador is not None
    modelo.train(treinando)
    perda_total, acertos, n = 0.0, 0, 0
    with torch.set_grad_enabled(treinando):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            saida = modelo(x)
            perda = criterio(saida, y)
            if treinando:
                otimizador.zero_grad(); perda.backward(); otimizador.step()
            perda_total += perda.item() * len(y)
            acertos += (saida.argmax(1) == y).sum().item()
            n += len(y)
    return perda_total / n, acertos / n

criterio = nn.CrossEntropyLoss()
otimizador = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=EPOCAS)

historico = []
melhor_acc, melhor_estado, sem_melhora = 0.0, None, 0
caminho_best = os.path.join(PASTA_SAIDA, f"{MODELO}_best.pt")
t0 = time.time()

for ep in range(1, EPOCAS + 1):
    congelar_backbone(modelo, cabeca, ep <= EPOCAS_CABECA)
    l_tr, a_tr = rodar_epoca(modelo, dl_treino, criterio, otimizador)
    l_va, a_va = rodar_epoca(modelo, dl_val, criterio)
    scheduler.step()
    historico.append(dict(epoca=ep, loss_treino=l_tr, acc_treino=a_tr, loss_val=l_va, acc_val=a_va))
    marca = ""
    if a_va > melhor_acc:
        melhor_acc, sem_melhora = a_va, 0
        melhor_estado = copy.deepcopy(modelo.state_dict())
        torch.save(melhor_estado, caminho_best)
        marca = "  <-- melhor"
    else:
        sem_melhora += 1
    print(f"Época {ep:02d}/{EPOCAS} | treino loss {l_tr:.4f} acc {a_tr:.4f} | val loss {l_va:.4f} acc {a_va:.4f}{marca}")
    if sem_melhora >= PACIENCIA:
        print(f"Early stopping (sem melhora há {PACIENCIA} épocas)."); break

tempo_treino = time.time() - t0
print(f"\nTreino concluído em {tempo_treino/60:.1f} min | melhor acc val: {melhor_acc:.4f}")
modelo.load_state_dict(melhor_estado)

In [ ]:
h = pd.DataFrame(historico)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoca, h.loss_treino, label="treino"); ax[0].plot(h.epoca, h.loss_val, label="validação")
ax[0].set_title("Loss"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(h.epoca, h.acc_treino, label="treino"); ax[1].plot(h.epoca, h.acc_val, label="validação")
ax[1].set_title("Acurácia"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(PASTA_SAIDA, f"{MODELO}_curvas.png"), dpi=150); plt.show()

## Salvando metadados
Guarda as informações necessárias para o notebook de comparação reconstruir o modelo.

In [ ]:
meta = dict(modelo=MODELO, img_size=IMG_SIZE, classes=CLASSES, melhor_acc_val=melhor_acc,
            epocas_treinadas=len(historico), tempo_treino_min=tempo_treino/60, n_params=n_params)
with open(os.path.join(PASTA_SAIDA, f"{MODELO}_meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
h.to_csv(os.path.join(PASTA_SAIDA, f"{MODELO}_historico.csv"), index=False)
print("Salvo em:", os.path.abspath(PASTA_SAIDA))
print(json.dumps(meta, ensure_ascii=False, indent=2))